Collecting workspace information

# Diagnosis: Why Two-Stage Cascade Performs Poorly

## A) File-Accurate Execution Trace of Two-Stage Inference

The cascade runs in `main()` in run_cascade.py. Here is the exact order:

### Step 1: Load PatchCore artifacts


In [ ]:
memory = np.load(patchcore_dir / "memory.npy")
source_threshold = json.load(patchcore_dir / "threshold.json")["threshold"]
backbone = PatchCoreBackbone(cfg["patchcore"]["backbone"])

Memory was built from **Severstal normal patches** in `train_patchcore.main()`.

### Step 2: Load classifier + kappa


In [ ]:
model = build_resnet50(num_classes=len(known_classes))
model.load_state_dict(ckpt["model_state"])
kappa = json.load(kappa_path)["kappa"]   # from osr/thresholds.json



### Step 3: Prepare NEU test data


In [ ]:
samples = collect_neu_samples(neu_root, class_mapping, splits, images_dir=images_dir)
known_samples = [s for s in samples if s[1] in known_classes]
unknown_samples = [s for s in samples if s[1] not in known_classes]
_, known_val, known_test = stratified_split(known_samples, train_ratio=0.7, val_ratio=0.15, seed=...)



**Critical**: The transform applied is:


In [ ]:
tf = transforms.Compose([
    transforms.Resize((cfg["image_size"], cfg["image_size"])),  # 224x224
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

[direct from code]

### Step 4: Tau calibration (target_known_val path)


In [ ]:
if tau_source == "target_known_val":
    known_val_scores = infer_anomaly_scores(known_val_loader, device, backbone, memory)
    threshold = float(calibrate_anomaly_threshold(known_val_scores, tau_accept_rate))

Where [`calibrate_anomaly_threshold`](src/patchcore/calibrate.py) delegates to [`calibrate_threshold`](src/osr/calibrate.py):


In [ ]:
def calibrate_threshold(scores, accept_rate):
    k = int(np.ceil(len(scores) * accept_rate))
    sorted_scores = np.sort(scores)
    tau = sorted_scores[k - 1]
    return float(tau)

This sets tau so that `accept_rate` fraction of scores fall **at or below** tau. [direct from code]

### Step 5: Score PatchCore on test sets


In [ ]:
known_scores = infer_anomaly_scores(known_loader, device, backbone, memory)
unk_scores = infer_anomaly_scores(unk_loader, device, backbone, memory)

[`infer_anomaly_scores`](src/patchcore/patchcore_simple.py) computes per-image `anomaly_score` = max nearest-neighbor L2 distance across all spatial patches.

### Step 6: Classifier confidences


In [ ]:
def get_confidences(loader):
    logits = model(x)
    conf = np.max(softmax(logits.cpu().numpy()), axis=1)



### Step 7: Cascade decision (THE CRITICAL LOGIC)


In [ ]:
if kappa is None:
    kappa = 0.0

known_no_defect = known_scores <= threshold      # Stage-1 says "normal"
unk_no_defect = unk_scores <= threshold           # Stage-1 says "normal"

known_reject = (~known_no_defect) & (conf_known < kappa)    # ← REJECT decision
unk_reject = (~unk_no_defect) & (conf_unknown < kappa)      # ← REJECT decision

[direct from code, lines in `run_cascade.py`]

### Step 8: Metrics


In [ ]:
tpr_unknown_system = float(np.mean(unk_reject))      # fraction of unknowns rejected
fpr_known_system = float(np.mean(known_reject))       # fraction of knowns wrongly rejected
stage1_pass_rate_known = float(np.mean(known_mask))   # where known_mask = ~known_no_defect
stage1_pass_rate_unknown = float(np.mean(unk_mask))



---

## B) Top 5 Root-Cause Candidates (Ranked by Likelihood)

### 🥇 Candidate 1: Domain mismatch causes PatchCore to reject almost everything (both known AND unknown NEU images)

**Evidence** [direct from code]:

- Memory bank is built from **Severstal steel strip normal patches** in [`train_patchcore.py`](src/pipelines/two_stage/train_patchcore.py) using `collect_severstal_normal_images()` from severstal.py.
- The PatchCore backbone (`PatchCoreBackbone`) extracts mid-level features (up to `layer3` of ResNet18).
- NEU images are an entirely different visual domain (different texture, resolution, surface type).
- The [`anomaly_score`](src/patchcore/patchcore_simple.py) function returns `max(min_L2_distance)` across spatial patches — **every** NEU image will have high anomaly scores relative to Severstal normals.

**Why this explains the sweep behavior** [inference from measured outputs]:

When tau is calibrated on **NEU known-val** PatchCore scores (which are all high due to domain shift), the threshold tau itself becomes very high. But the *relative* separation between known and unknown NEU scores is tiny, because **both** are far from the Severstal memory bank. The pass rates at `accept_rate=0.95` are only ~6% for both known and unknown — meaning ~94% of **all** NEU images are classified as "no defect" (score ≤ tau). This is backwards: nearly all NEU defect images get **leaked** at Stage 1.

Wait — let me re-read the logic carefully:



In [ ]:
known_no_defect = known_scores <= threshold



If tau is calibrated on NEU known-val scores at 95%, then 95% of known-val scores are ≤ tau. So for known-test (similar distribution), ~95% will also have `scores <= threshold`, meaning `known_no_defect ≈ True` for 95%. Therefore:



In [ ]:
stage1_pass_rate_known = mean(~known_no_defect) ≈ 0.05



This matches the measured **0.0599**. The 95% of known images are being **leaked as "no defect"** and never reach Stage 2. [direct from code + measured outputs confirm]

**The fundamental problem**: `calibrate_threshold` at `accept_rate=0.95` means 95% of scores are **accepted as normal**. But NEU images are ALL defective. So 95% of defective images are being called normal. The threshold semantics are inverted for this use case.

In the Severstal context (where calibration was designed), accepting 95% of **normal** patches makes sense — you want most normals to pass as normal. But when you recalibrate on **NEU known defect images**, accepting 95% means **95% of defect images are called normal**, which is catastrophic.

**Likelihood: ★★★★★ (near-certain primary cause)**

---

### 🥈 Candidate 2: The reject decision requires BOTH conditions — AND gate is too strict

**Evidence** [direct from code]:


In [ ]:
known_reject = (~known_no_defect) & (conf_known < kappa)
unk_reject = (~unk_no_defect) & (conf_unknown < kappa)



To be rejected as unknown, a sample must:
1. Pass Stage 1 (`score > threshold`) — already only ~5-6% do
2. AND have low classifier confidence (`conf < kappa`)

Even among the ~5% that pass Stage 1, many will have `conf >= kappa` and thus NOT be rejected. This double-gate makes system-level TPR vanishingly small.

**Why this explains the sweep**: Even at `accept_rate=0.60` (where ~37% pass Stage 1), `tpr_unknown_system` is only 0.051 because the confidence gate further filters. [direct from code]

**Likelihood: ★★★★☆ (confirmed secondary bottleneck)**

---

### 🥉 Candidate 3: Transform mismatch — PatchCore training uses Grayscale but cascade inference does not

**Evidence** [direct from code]:

In `train_patchcore.py`, the Severstal training transform is:


In [ ]:
tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # ← GRAYSCALE
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])



In `run_cascade.py`, the NEU inference transform is:


In [ ]:
tf = transforms.Compose([
    transforms.Resize((cfg["image_size"], cfg["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])



**No `Grayscale` and no `Resize`** in cascade! The memory bank was built from grayscale-converted, non-resized Severstal patches (which are already 224×224 from the patch extraction). NEU images are being fed as full-color and resized to 224×224. This means:

1. **Color channel mismatch**: Memory features are from grayscale(→3ch) images; query features are from RGB images. The ResNet18 backbone will produce different feature distributions.
2. **Missing Resize in training**: Actually, Severstal patches are cropped to `patch_size=224` by [`SeverstalPatchDataset.__getitem__`](src/datasets/severstal.py) which does `img.crop(...)`, so they're already 224×224. But NEU images are resized from their native resolution. This is less of an issue since both end up at 224×224.

The **Grayscale mismatch is significant**: it means PatchCore features in memory occupy a different subspace than query features from RGB NEU images, making all distances artificially large and reducing discriminative power.

**Likelihood: ★★★★☆ (confirmed contributing factor, amplifies Candidate 1)**

---

### Candidate 4: `kappa` defaults to 0.0 when thresholds.json doesn't exist yet

**Evidence** [direct from code]:


In [ ]:
kappa_path = os.path.join(out_dir, "osr", "thresholds.json")
if os.path.exists(kappa_path):
    kappa = json.load(open(kappa_path, "r")).get("kappa")
else:
    kappa = None
# ...
if kappa is None:
    kappa = 0.0



If the one-stage OSR pipeline has already run, `kappa` is loaded from `thresholds.json`. But if `use_confidence_gate` is True (the default), kappa is typically a positive value like 0.5–0.9. If it hasn't run, `kappa=0.0`, meaning `conf < 0.0` is never true and the confidence gate passes everything through (no rejections from confidence alone). However, since the observed runs show the OSR pipeline **has** been run (osr metrics exist), this is likely not the issue in the measured results.

If kappa **is** loaded and is high (e.g., 0.95), then `conf < kappa` is true for many samples, but the AND with Stage-1 pass still limits the effect.

**Likelihood: ★★☆☆☆ (situational)**

---

### Candidate 5: Severstal memory bank has no discriminative power for NEU defect types

**Evidence** [direct from code]:

The `anomaly_score` function:


In [ ]:
def anomaly_score(patches, memory):
    dists = []
    for p in patches:
        diff = memory - p
        dd = np.sqrt(np.sum(diff * diff, axis=1))
        dists.append(np.min(dd))
    return float(np.max(dists))



This computes max-over-patches of min-nearest-neighbor L2. For **all** NEU images (both known and unknown defect types), the distances to Severstal memory will be uniformly large. The score distributions for known-NEU and unknown-NEU will be heavily overlapping, so **no threshold can separate them**.

This is the information-theoretic version of Candidate 1: even if the threshold semantics were correct, the Severstal memory bank simply cannot distinguish NEU known defects from NEU unknown defects.

**Likelihood: ★★★★★ (fundamental limitation, same root as Candidate 1)**

---

## C) Minimal Verification Checks

### For Candidate 1 (threshold semantics inversion):



In [ ]:
# Add to run_cascade.py after computing known_val_scores
# filepath: src/pipelines/two_stage/run_cascade.py

# DIAGNOSTIC: Check what fraction of known-val scores are below tau
print(f"[DIAG] known_val_scores: min={known_val_scores.min():.4f} "
      f"max={known_val_scores.max():.4f} mean={known_val_scores.mean():.4f}")
print(f"[DIAG] threshold (tau): {threshold:.4f}")
print(f"[DIAG] frac known_val <= tau: {np.mean(known_val_scores <= threshold):.4f}")
print(f"[DIAG] frac known_test <= tau (leaked as no_defect): {np.mean(known_scores <= threshold):.4f}")
print(f"[DIAG] frac unknown_test <= tau (leaked as no_defect): {np.mean(unk_scores <= threshold):.4f}")



**Expected output confirming the bug**: `frac known_val <= tau ≈ 0.95`, `frac known_test <= tau ≈ 0.94`, `frac unknown_test <= tau ≈ 0.94`.

### For Candidate 2 (AND gate):



In [ ]:
# Add after computing reject decisions
print(f"[DIAG] Stage1 pass (known): {np.sum(~known_no_defect)}/{len(known_no_defect)}")
print(f"[DIAG] Stage1 pass (unknown): {np.sum(~unk_no_defect)}/{len(unk_no_defect)}")
print(f"[DIAG] Of those, conf < kappa (known): {np.sum((~known_no_defect) & (conf_known < kappa))}")
print(f"[DIAG] Of those, conf < kappa (unknown): {np.sum((~unk_no_defect) & (conf_unknown < kappa))}")



### For Candidate 3 (Grayscale mismatch):



In [ ]:
# Quick check: compare feature norms between grayscale and RGB for same image
from torchvision import transforms
from PIL import Image

tf_gray = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tf_rgb = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

img = Image.open("some_neu_image.bmp").convert("RGB")
feat_gray = backbone(tf_gray(img).unsqueeze(0).to(device))
feat_rgb = backbone(tf_rgb(img).unsqueeze(0).to(device))
print(f"Feature L2 norm (gray): {feat_gray.norm():.4f}")
print(f"Feature L2 norm (rgb): {feat_rgb.norm():.4f}")
print(f"Feature L2 distance gray vs rgb: {(feat_gray - feat_rgb).norm():.4f}")



### For Candidate 5 (score distribution overlap):



In [ ]:
# Add after computing known_scores and unk_scores
from scipy.stats import ks_2samp
stat, pval = ks_2samp(known_scores, unk_scores)
print(f"[DIAG] KS test known vs unknown PatchCore scores: stat={stat:.4f} p={pval:.4f}")
print(f"[DIAG] known_scores: mean={np.mean(known_scores):.4f} std={np.std(known_scores):.4f}")
print(f"[DIAG] unk_scores: mean={np.mean(unk_scores):.4f} std={np.std(unk_scores):.4f}")



**Expected**: p-value will be high (distributions not significantly different), confirming PatchCore cannot separate known from unknown NEU defects.

---

## D) Minimal Patches to Address Highest-Likelihood Causes

### Patch 1: Fix threshold semantics — invert the Stage-1 gate direction

The core issue is that `score <= threshold` means "accepted as normal." For NEU (all-defect domain), we want Stage 1 to **pass through** images that look anomalous (high score), not reject them. The `accept_rate` when calibrated on NEU known-val should mean "fraction that **passes to Stage 2**" (i.e., are anomalous enough), not "fraction accepted as normal."



In [ ]:
// ...existing code...
    tau_source = two_stage_cfg.get("tau_source", "source_patchcore")
    tau_accept_rate = float(two_stage_cfg.get("tau_accept_rate", cfg["patchcore"]["accept_rate"]))
    threshold = float(source_threshold)
    known_val_scores = None
    if tau_source == "target_known_val":
        known_val_scores_path = os.path.join(cascade_dir, "stage1_known_val_scores.npy")
        tau_info_path = os.path.join(cascade_dir, "stage1_tau.json")
        if os.path.exists(known_val_scores_path):
            known_val_scores = np.load(known_val_scores_path)
            print(f"[cascade:{split_name}] loaded cached known-val PatchCore scores")
        else:
            print(f"[cascade:{split_name}] scoring PatchCore for known val (tau calibration)...")
            known_val_scores = infer_anomaly_scores(known_val_loader, device, backbone, memory)
            np.save(known_val_scores_path, known_val_scores)
            print(f"[cascade:{split_name}] saved known-val scores: {known_val_scores_path}")
        # Use (1 - tau_accept_rate) as the accept_rate so that tau_accept_rate
        # fraction of known-val scores fall ABOVE the threshold (i.e. pass Stage 1).
        threshold = float(calibrate_anomaly_threshold(known_val_scores, 1.0 - tau_accept_rate))
        save_json(tau_info_path, {
            "tau_source": "target_known_val",
            "tau_accept_rate": tau_accept_rate,
            "effective_normal_accept_rate": 1.0 - tau_accept_rate,
            "threshold": threshold,
            "source_patchcore_threshold": float(source_threshold),
            "known_val_count": int(len(known_val_scores)),
        })
        print(
            f"[cascade:{split_name}] calibrated tau from target known val: "
            f"{threshold:.6f} (pass_rate={tau_accept_rate}, i.e. {tau_accept_rate*100:.0f}% of known pass Stage 1)"
        )
// ...existing code...



**Explanation**: Currently `calibrate_threshold(scores, 0.95)` returns the 95th percentile, so 95% of scores are ≤ tau → 95% are called "no defect." By passing `1.0 - 0.95 = 0.05`, we get the 5th percentile, so only 5% are called "no defect" and **95% pass to Stage 2**. 

**Important**: Delete cached `stage1_known_val_scores.npy` and `stage1_tau.json` files before rerunning, since the scores themselves are fine but the threshold file will be stale.

### Patch 2: Fix the Grayscale transform mismatch



In [ ]:
// ...existing code...
    tf = transforms.Compose([
        transforms.Resize((cfg["image_size"], cfg["image_size"])),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
// ...existing code...



This ensures NEU images go through the same grayscale conversion as Severstal patches did during memory bank construction in `train_patchcore.py`. Without this, the feature-space distances are inflated by the color channel discrepancy.

### Patch 3: Relax the AND gate to OR for the reject decision



In [ ]:
// ...existing code...
    known_mask = ~known_no_defect
    unk_mask = ~unk_no_defect

    # Reject as unknown if EITHER stage flags the sample:
    # - Stage 1 flags it as anomalous AND classifier is unconfident, OR
    # - Stage 1 passes it but classifier confidence is very low
    # For backward compatibility, also compute the AND-gated versions.
    known_reject_and = (~known_no_defect) & (conf_known < kappa)
    unk_reject_and = (~unk_no_defect) & (conf_unknown < kappa)

    known_reject_or = (~known_no_defect) | (conf_known < kappa)
    unk_reject_or = (~unk_no_defect) | (conf_unknown < kappa)

    # Use OR gate as the primary decision
    known_reject = known_reject_or
    unk_reject = unk_reject_or
// ...existing code...



However, **Patch 1 alone should be the priority** — the OR gate change is secondary and changes the method semantics more significantly.

### Patch 4: Clear stale cache before rerunning



In [ ]:
"""Run this once before re-executing the cascade with fixed threshold logic."""
import os
import glob

output_base = "outputs"  # or "/content/drive/MyDrive/fyp_outputs" for Colab

for split in ["split_a", "split_b", "split_c"]:
    cascade_dir = os.path.join(output_base, split, "cascade")
    for pattern in ["stage1_tau.json", "stage1_known_val_scores.npy"]:
        path = os.path.join(cascade_dir, pattern)
        if os.path.exists(path):
            os.remove(path)
            print(f"Removed: {path}")
    # Also remove cached metrics
    for f in ["metrics.json"]:
        path = os.path.join(cascade_dir, f)
        if os.path.exists(path):
            os.remove(path)
            print(f"Removed: {path}")



---

## E) Rejected Prior Assumptions

| Assumption | Status | Evidence |
|---|---|---|
| "There may be a score inversion bug (using `<` instead of `>`)" | **Rejected** | The code uses `known_scores <= threshold` for "no defect" and `~known_no_defect` for "pass to Stage 2" — the **direction is correct**. The bug is in **where tau is set**, not the comparison operator. [direct from code] |
| "The confidence gate might be disabled" | **Rejected** | `use_confidence_gate: true` is default in default.yaml, and `kappa` is loaded from `osr/thresholds.json` which exists after the one-stage pipeline runs. [direct from code] |
| "Cache reuse might load wrong split's data" | **Rejected** | `cascade_dir` is per-split (`outputs/split_X/cascade/`), and cached files are named consistently. Split name normalization for `.colab` configs is handled by `split_name_from_config()`. [direct from code] |
| "The memory bank might be too small after coreset" | **Partially rejected** | While `coreset_sampling_ratio=0.1` does reduce the bank, the fundamental issue is cross-domain mismatch, not bank size. Even the full pre-coreset bank would have the same NEU distribution overlap problem. [inference from measured outputs] |
| "`calibrate_threshold` might have an off-by-one error" | **Rejected** | `k = int(np.ceil(len(scores) * accept_rate))` then `tau = sorted_scores[k-1]` is correct for the k-th smallest value. The issue is that `accept_rate=0.95` means 95% of scores are **below** tau, which has the wrong semantic meaning for the cascade use case on NEU. [direct from code] |

---

## Summary

The **primary root cause** is a **threshold semantics mismatch**: [`calibrate_anomaly_threshold`](src/patchcore/calibrate.py) (via [`calibrate_threshold`](src/osr/calibrate.py)) sets tau so that `accept_rate` fraction of scores are **at or below** it — meaning they're classified as "normal/no-defect." When this is applied to NEU known-validation scores with `accept_rate=0.95`, it causes **95% of all NEU defect images to be leaked as "no defect" at Stage 1**, leaving only ~5% to reach Stage 2. This directly explains the observed `stage1_pass_rate ≈ 0.06` at `accept_rate=0.95` and the monotonic improvement as `accept_rate` decreases.

**Patch 1** (inverting the accept_rate to `1.0 - tau_accept_rate`) is the minimal, highest-impact fix. **Patch 2** (adding Grayscale to cascade transforms) addresses a real preprocessing inconsistency. Together, these should dramatically improve pass rates and system-level TPR.